In [2]:
import shutil
from functools import reduce
from pathlib import Path
from time import sleep

import pyarrow as pa
import pyarrow.parquet as pq

import numpy as np
import polars as pl
from tqdm import tqdm

In [3]:
DATA_DIR = Path("/mnt/Fast2T/data/ai-training-set/")
DATASET_DIR = Path(DATA_DIR, "dataset/")

Find embedding
We embedd a text using the usage of stem words, so to find a embedding we combine the top words of:
- web data (to have some relation to our data set)
- ai data (to find patterns like —)
- wikipedia data (to find human patterns)

In [3]:
TOP_K = 2048

ai = pl.read_parquet(DATA_DIR / "total-words" / "ai.parquet")
wiki = pl.read_parquet(DATA_DIR / "total-words" / "wiki.parquet")
website = pl.read_parquet(DATA_DIR / "total-words" / "website.parquet")

common = (
    ai.select("term", (pl.col("len") / pl.col("len").sum()).alias("len_ai"))
    .join(ai.select("term", (pl.col("len") / pl.col("len").sum()).alias("len_wiki")), on="term")
    .join(ai.select("term", (pl.col("len") / pl.col("len").sum()).alias("len_website")), on="term")
    .unique(subset="term")
)

embedding = (
    common
    .with_columns(pl.mean_horizontal("len_ai", "len_wiki", "len_website").alias("len_avg"))
    .top_k(k=TOP_K, by="len_avg")
    .sort("term")
    .select("term")
    .rename({"term": "token"})
    .with_row_index("id")
)

embedding.write_parquet(DATASET_DIR / "embedding.parquet")

embedding_size = int(embedding.count()["id"][0])
embedding

id,token
u32,str
0,"""__name__"""
1,"""ability"""
2,"""able"""
3,"""about"""
4,"""above"""
…,…
2043,"""you’re"""
2044,"""z"""
2045,"""zero"""


Create a dataset based on the embedding

In [4]:
def to_np_arr(weight, id_right):
    vec = np.zeros(embedding_size, dtype=np.float32)
    vec[id_right] = weight
    return vec


def apply_embedding(source_dir: Path, prefix: str, flag: int):
    file_idx = 0

    for file in tqdm(list((DATA_DIR / source_dir).iterdir())):
        if file.suffix != ".parquet":
            continue

        weights = (pl.scan_parquet(file)
                   .select("id", "term", "len")
                   .join(embedding.lazy(), left_on="term", right_on="token", how="inner")
                   .with_columns(total=pl.col("len").sum().over("id"))
                   .with_columns(weight=(pl.col("len") / pl.col("total")).cast(pl.Float32))
                   .select("id", "weight", "id_right")
                   .collect())

        # @author Claude (Claude recommended using matrix instead of my join mess and it worked)
        ids = weights["id"].unique().sort().to_frame().with_row_index("row")
        weights = weights.join(ids, on="id")

        matrix = np.zeros((len(ids), embedding_size), dtype=np.float32)
        matrix[weights["row"].to_numpy(), weights["id_right"].to_numpy()] = weights["weight"].to_numpy()

        pq.write_table(
            pa.table({
                "embedding": pa.FixedSizeListArray.from_arrays(matrix.ravel(), embedding_size),
                "flag": pa.array(np.full(len(ids), flag, dtype=np.int32)),
            }),
            DATASET_DIR / "partial" / f"{prefix}-{file_idx}.parquet",
            compression="zstd",
        )

        file_idx += 1


shutil.rmtree(DATASET_DIR / "partial")

(DATASET_DIR / "partial").mkdir()

apply_embedding(DATA_DIR / "exploded-wiki", "wiki", 0)
apply_embedding(DATA_DIR / "exploded-ai", "ai", 1)
# apply_embedding(DATA_DIR / "exploded-website", "web", 2)

100%|██████████| 109/109 [04:56<00:00,  2.72s/it]


In [4]:
# @author Claude
def sink_partitioned(source_glob, output_dir, name, target=1_000_000):
    partial_files = sorted(Path(source_glob).parent.glob(Path(source_glob).name))

    buffer = []
    file_index = 0
    buffer_size = 0

    for f in partial_files:
        df = pl.read_parquet(f)
        buffer.append(df)
        buffer_size += len(df)

        while buffer_size >= target:
            combined = pl.concat(buffer)
            combined[:target].write_parquet(
                output_dir / f"{name}-{file_index}.parquet",
                compression="zstd"
            )
            file_index += 1
            rest = combined[target:]
            buffer = [rest] if len(rest) > 0 else []
            buffer_size = len(rest)

    if buffer:
        pl.concat(buffer).write_parquet(
            output_dir / f"{name}-{file_index}.parquet",
            compression="zstd"
        )

    print(f"{name}: wrote {file_index + 1} files")


sink_partitioned(
    DATASET_DIR / "partial" / "ai-*.parquet",
    DATASET_DIR / "complete", "ai"
)
sink_partitioned(
    DATASET_DIR / "partial" / "wiki-*.parquet",
    DATASET_DIR / "complete", "wiki"
)

ai: wrote 10 files
wiki: wrote 11 files
